# Search effort and solve time across resolutions

How much harder does an exact search become as the grid grows from 64×64 to 4096×4096 cells? This
notebook answers with two measures. The first is recorded in the data: the number of cells the
generation search considered for each stored route. The second is measured here: the warm solve time
of the canonical A\* planner of `passage-eval` on a few tasks per resolution, on the machine that runs
the notebook.

The times describe this run only. They are not a benchmark of A\*, nor a reference for learned planners.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It reads a few thousand metadata records and solves a few dozen tasks; about five minutes.

In [ ]:
resolutions = [64, 128, 256, 512, 1024, 2048, 4096]
split = "validation"
metadata_sample = 256  # validation scenes per resolution read in Section 1
timed_tasks = {64: 16, 128: 16, 256: 16, 512: 16, 1024: 8, 2048: 4, 4096: 4}
objective = "elevation"
condition = "obstacles"
output_dir = "notebook-artifacts/notebooks/paper/solver_runtime_scaling"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2ab44d899652c7fb99c425488e84a7d565963547"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))

## 1. Recorded search effort

Each stored route records `considered_area`: the number of cells that the generation search marked as
considered before it closed the route. Figure 1 compares it with the number of cells of the grid.

In [ ]:
SOLVER = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}[objective]
records = []
for resolution in resolutions:
    for row in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False, limit=metadata_sample):
        path = row["metadata"]["paths"][SOLVER][condition]
        if path["status"] == "ok":
            records.append({"resolution": resolution, "cells": resolution ** 2, "considered": path["considered_area"]})
effort = pd.DataFrame(records)
effort_table = effort.groupby("resolution").considered.describe(percentiles=[0.1, 0.5, 0.9])[["count", "10%", "50%", "90%"]]
effort_table["Median share of grid (%)"] = 100 * effort_table["50%"] / effort_table.index.to_series() ** 2
effort_table = effort_table.rename(columns={"count": "Routes", "10%": "10th percentile", "50%": "Median",
                                            "90%": "90th percentile"}).reset_index(names="Grid side")
show(effort_table, f"Table 1. Cells considered by the generation search ({OBJECTIVE_LABELS[objective]}, {condition}).", "{:,.0f}")

## 2. Measured solve time

Tasks are decoded before timing. Each task gets one untimed call, which also compiles the search on the
first task, then three timed calls; the task time is their median.

In [ ]:
import time
from passage_eval.io.dataset import scene_from_row
from passage_eval.baselines.classical import AStarPlanner

planner = AStarPlanner()
timings = []
for resolution in resolutions:
    for row in source.iter_rows(resolution, split, columns=None, samples=False, limit=timed_tasks[resolution]):
        if row["metadata"]["paths"][SOLVER][condition]["status"] != "ok":
            continue  # only an exact reference route is timed; every released variant is "ok"
        task = scene_from_row(row, resolution=resolution, cost_model=objective, split=split, task_variant=condition)
        planner.predict_path(task)
        samples = []
        for _ in range(3):
            began = time.perf_counter()
            planner.predict_path(task)
            samples.append(time.perf_counter() - began)
        timings.append({"resolution": resolution, "cells": resolution ** 2, "seconds": float(np.median(samples))})
timings = pd.DataFrame(timings)
time_table = timings.groupby("resolution").seconds.agg(["size", "median", "max"]).reset_index()
time_table.columns = ["Grid side", "Tasks", "Median (s)", "Slowest (s)"]
show(time_table, "Table 2. Warm solve time of the canonical A* planner on this machine.", "{:,.4f}")
exponent = np.polyfit(np.log(timings.cells), np.log(timings.seconds), 1)[0]

fig, (ax_effort, ax_time) = plt.subplots(1, 2, figsize=(11, 3.9))
fig.subplots_adjust(top=0.78, wspace=0.25)
med = effort.groupby("cells").considered.median()
ax_effort.plot(med.index, med.values, marker="o", markersize=5, color="#1767ac", label="Median considered")
ax_effort.plot(med.index, med.index, color=MUTED, linestyle=":", linewidth=1, label="All cells of the grid")
ax_time.scatter(timings.cells, timings.seconds, s=16, color="#238b45", alpha=0.8, linewidths=0, label="One task")
fit = np.exp(np.polyval(np.polyfit(np.log(timings.cells), np.log(timings.seconds), 1), np.log(med.index)))
ax_time.plot(med.index, fit, color=INK, linewidth=1, linestyle="--", label=f"Fit: time ∝ cells^{exponent:.2f}")
for ax, ylabel, title in ((ax_effort, "Cells considered", "Recorded search effort"),
                          (ax_time, "Warm solve time (s)", "Measured solve time")):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set(xlabel="Cells in the grid", ylabel=ylabel, title=title)
    ax.grid(True, axis="both")
    ax.legend(fontsize=8)
figure_title(fig, "Exact search cost grows with the number of cells",
             f"{OBJECTIVE_LABELS[objective]} objective · {condition} · {split} split · log scales")
figures = save_figure(fig, output_path, f"solver_runtime_scaling-{objective}")
plt.show()

*Figure 1. Left: median number of cells considered by the generation search; the dotted line marks every
cell of the grid. Right: warm solve time of each timed task, with a power-law fit.*

## Summary

In [ ]:
display(Markdown("\n".join([
    f"- The generation search considers a median of {effort_table['Median share of grid (%)'].min():.0f} to "
    f"{effort_table['Median share of grid (%)'].max():.0f} % of the grid, depending on the resolution.",
    f"- On this machine, the warm A\\* solve time grows roughly as the number of cells to the power {exponent:.2f}, "
    f"from {time_table['Median (s)'].iloc[0] * 1000:.1f} ms at {resolutions[0]}×{resolutions[0]} to "
    f"{time_table['Median (s)'].iloc[-1]:.1f} s at {resolutions[-1]}×{resolutions[-1]} (medians).",
    "- Large grids therefore dominate the cost of exact labels, and of any planner that falls back on exact search.",
])))
report = {"objective": objective, "condition": condition, "effort": effort_table.to_dict("records"),
          "timings": time_table.to_dict("records"), "time_exponent": float(exponent),
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"solver_runtime_scaling-{objective}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*